# Diagnose a Problematic ML Pipeline

In this exercise, you will implement a series of steps towards diagnosing bias in an ML pipeline with the bias identification methdologies we identified, using the German Credit Dataset provided by the AIF360 toolkit. We have included a few helper code blocks for loading the dataset and generating fairness metrics from [the AIF360 tutorials](https://github.com/Trusted-AI/AIF360/blob/master/examples/tutorial_credit_scoring.ipynb) that you will be building on for this simple analysis.

In [ ]:
# Load all necessary packages
import sys
sys.path.insert(1, "../../")

import numpy as np
np.random.seed(0)

from aif360.datasets import GermanDataset
from aif360.metrics import BinaryLabelDatasetMetric
from aif360.algorithms.preprocessing import Reweighing

from IPython.display import Markdown, display
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

## Load Data

Below, we load and setup the data, defining our privileged class as male and our label and protected attribute maps.

In [ ]:
#Helper code

#Define the label map for the dataset
label_map = {1.0: 'Good Credit', 2.0: 'Bad Credit'}

#Define the protected attribute for the dataset
protected_attribute_maps = [{1.0: 'Male', 0.0: 'Female'}]

#Load data, defining privilege class as male
gd = GermanDataset(protected_attribute_names=['sex'],
privileged_classes=[['male']], metadata={'label_map': label_map,
'protected_attribute_maps': protected_attribute_maps})

df, df_dict = gd.convert_to_dataframe()
viz_df = df.copy()

In [ ]:
#Visualize the dataframe
viz_df.head()

## Analyze the representation of the dataset 

Using `viz_df`, create two bar plots showing the representation of gender and credit in the dataset. Briefly interpret your findings - which categories seem to have a higher number of instances?

In [ ]:
viz_df['sex'] = viz_df['sex'].map(protected_attribute_maps[0])
ax = viz_df.sex.value_counts().plot(kind = 'bar')
ax.set_title("Representation of Gender in GermanDataset")
ax.set_xlabel("sex")
ax.set_ylabel("Number of instances")
plt.show()

In [ ]:
viz_df['credit'] = viz_df['credit'].map(label_map)
ax = viz_df.credit.value_counts().plot(kind = 'bar')
ax.set_title("Representation of Credit in GermanDataset")
ax.set_xlabel("credit")
ax.set_ylabel("Number of instances")
plt.show()

Analyze the representation of credit, stratified by gender, in the dataset. You can use the pandas groupby() operation to do so. Add a brief interpretation.

In [ ]:
ax = viz_df.groupby('credit').sex.value_counts().unstack().plot.bar()
ax.set_title("Representation of Credit Scores, per Gender, in German Dataset")
ax.set_xlabel("Credit")
ax.set_ylabel("Number of instances")
plt.show()

Interpretation: We see a higher represention of men in the dataset, both for the bad and good credit categories. We see a higher representation of men with good credit scores compared to women, which could cause a bias in the dataset.

## Correlation Matrix

Create a correlation matrix with the `sex`, `age`, and `credit` variables. Interpret the correlations - do you see any strong relationships.

In [ ]:
corr = df[['sex', 'age', 'credit', 'credit_amount']].corr()
corr.style.background_gradient(cmap='coolwarm')

We do not see a strong correlation between our target variable, credit, and sample protected attributes, age and sex, which is a good thing.

---
## Confusion Matrix

Next, you'll be asked to create and train a simple Logistic Regression model on the dataset. We've created the training and test data split and applied scaling to the dataset for you.

In [ ]:
#Split the dataset into training and test datasets
dataset_orig_train, dataset_orig_test = gd.split([0.7], shuffle=True)

#Define priviledge and unprivileged groups
privileged_groups = [{'sex': 1}]
unprivileged_groups = [{'sex': 0}]

#Apply scaling to the dataset
scale_orig = StandardScaler()
X_train = scale_orig.fit_transform(dataset_orig_train.features)
y_train = dataset_orig_train.labels.ravel()
w_train = dataset_orig_train.instance_weights.ravel()

Implement and train a simpler logistic regression sklearn model (with all defaults). Generate predictions on the training dataset.

In [ ]:
lmod = LogisticRegression()
lmod.fit(X_train, y_train, 
         sample_weight=dataset_orig_train.instance_weights)
y_train_pred = lmod.predict(X_train)

Plot a confusion matrix, and intepret the failures of the model. Which is higher for the model - false negatives or false positives?

In [ ]:
cm = confusion_matrix(y_train, y_train_pred, labels=lmod.classes_)
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                              display_labels=lmod.classes_)
disp.plot()
plt.show()

In [ ]:
#Helper snipet for labeling axes of confusion matrix
#Source: https://stackoverflow.com/questions/71038269/how-label-to-the-data-not-the-axes-of-the-plot-of-a-confusion-matrix-that-disp
import seaborn as sns
sns.set(font_scale=3)
plt.figure(figsize=(7, 7))
ax = sns.heatmap(data=[[1, 0], [0, 1]], cmap=sns.color_palette(['tomato', 'lightgreen'], as_cmap=True),
                 annot=[[f"TP={cm[0][0]:.0f}", f"FP={cm[0][1]:.0f}"], [f"FN={cm[1][0]:.0f}", f"TN={cm[1][1]:.0f}"]],
                 fmt='', annot_kws={'fontsize': 30}, cbar=False, square=True)

ax.set_xlabel('Actual Values')
ax.set_ylabel('Predicted')
ax.tick_params(length=0, labeltop=True, labelbottom=False)
ax.xaxis.set_label_position('top')
ax.set_xticklabels(['Positive', 'Negative'])
ax.set_yticklabels(['Positive', 'Negative'], rotation=90, va='center')
ax.add_patch(plt.Rectangle((0, 1), 1, 0.1, color='yellow', clip_on=False, zorder=0, transform=ax.transAxes))
ax.add_patch(plt.Rectangle((0, 0), -0.1, 1, color='yellow', clip_on=False, zorder=0, transform=ax.transAxes))
plt.tight_layout()
plt.show()

Interpretation: We see a higher number of false negatives, implying the model is more likely to predict a bad credit score, that is actually positive credit.

***Extension:*** Generate confusion matrices for slices of your dataset, sliced on the protected attribute.

## Fairness Metrics

Let's conclude our diagnosis of bias in the pipeline with an analysis of statistical parity difference and disparate impact in the dataset. We've included the helper code for you below - interpret the findings.

In [ ]:
# Define privileged and unprivileged groups
privileged_groups = [{'sex': 1}]
unprivileged_groups = [{'sex': 0}]

#Define DatasetMetric class
metric_orig_train = BinaryLabelDatasetMetric(dataset_orig_train, 
                                             unprivileged_groups=unprivileged_groups,
                                             privileged_groups=privileged_groups)
display(Markdown("#### Original training dataset"))
#Print statstical parity difference and disparate impact metrics
print("Statistical Parity Difference between unprivileged and privileged groups = %f" % metric_orig_train.mean_difference())
print("Disparate Impact between unprivileged and privileged groups = %f" % metric_orig_train.disparate_impact())

Interpretation:

**Statistical Parity Difference: ** The metric is less than 0, indicating higher benefit for the privileged group  - men.

**Disparate Impact: ** The ideal value for this metric is a 1.0. 
The metric is less than 1, indiacting a higher benefit for the privileged group, men.